# Experiment 06 — ER=EPR Bridge Conductance

Validates **RBLE Eq. (7)** ER=EPR bridge conductance tensor and district master equation:

$$\frac{dV_i}{dt} = F(V_i) + \sum_j G_{ij}(V_j - V_i) + \mathcal{B}_i$$

Build $G_{ij}$ on a district graph, evolve voltages/potentials over time, plot cross-district coupling.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src" / "polomni").is_dir():
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import numpy as np
import matplotlib.pyplot as plt

try:
    import networkx as nx
except ImportError:
    nx = None

%matplotlib inline
plt.rcParams.update({"figure.figsize": (9, 5), "font.size": 11})
print(f"polomni root: {ROOT}")


## Build multi-district graph with choice branching


In [ ]:
from polomni.core.superspace.district_graph import DistrictGraph
from polomni.core.conductance.bridge_tensor import bridge_conductance, conductance_matrix
from polomni.core.conductance.master_equation import district_master_step

graph = DistrictGraph(gravity_mutation_strength=0.08)
root = graph.add_district(2.0, [1.0, 1.0], [0.0, 0.0, 0.0], 1e-120)
packets_a = graph.trigger_choice_event(root, 3)
child_a = packets_a[0].district_id
packets_b = graph.trigger_choice_event(child_a, 4)

n_nodes = graph.graph.number_of_nodes()
n_edges = graph.graph.number_of_edges()
print(f"Graph: {n_nodes} districts, {n_edges} portal edges")


## Assemble conductance matrix $G_{ij}$


In [ ]:
G = conductance_matrix(graph.graph)
assert G.shape == (n_nodes, n_nodes)
assert np.allclose(G, G.T, atol=1e-12)
assert np.all(G >= 0.0)

print("G_ij matrix:\n", np.round(G, 4))


## Pairwise bridge_conductance from stream packets


In [ ]:
pkt_i = packets_a[0]
pkt_j = packets_a[1]
g_pair = bridge_conductance(
    s_euclidean=0.5,
    stream_i=np.asarray(pkt_i.phi_stream),
    stream_j=np.asarray(pkt_j.phi_stream),
    t_munu_coupling=1.2,
)
assert g_pair >= 0.0
print(f"bridge_conductance(i,j) = {g_pair:.6f}")


## Master equation setup


In [ ]:
n = G.shape[0]
V = np.linspace(1.0, 0.2, n)
F = -0.1 * V  # relaxation drive
branch_kicks = np.zeros(n)
branch_kicks[1] = 0.3  # impulse at district 1
dt = 0.05
n_steps = 60

history = [V.copy()]
for _ in range(n_steps):
    V = district_master_step(V, F, G, branch_kicks, dt)
    history.append(V.copy())

history = np.array(history)
assert history.shape == (n_steps + 1, n)


## Verify coupling redistributes potential


In [ ]:
spread_initial = float(np.std(history[0]))
spread_final = float(np.std(history[-1]))
# Coupling should reduce variance vs uncoupled decay in some regimes
print(f"std(V) initial={spread_initial:.4f}, final={spread_final:.4f}")
assert np.all(np.isfinite(history))


## Plot cross-district coupling time series


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for i in range(n):
    ax.plot(history[:, i], label=f"district {list(graph.graph.nodes)[i]}", lw=2)
ax.set_xlabel("time step")
ax.set_ylabel("V_i")
ax.set_title("Master equation evolution with ER=EPR coupling (RBLE Eq. 7)")
ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()


## Heatmap of $G_{ij}$


In [ ]:
fig, ax = plt.subplots()
im = ax.imshow(G, cmap="YlOrRd")
nodes = list(graph.graph.nodes)
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(nodes)
ax.set_yticklabels(nodes)
ax.set_xlabel("j")
ax.set_ylabel("i")
ax.set_title("Conductance matrix G_ij")
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()


## Coupling flux between districts


In [ ]:
coupling_flux = []
for Vt in history:
    flux = G @ Vt - np.diag(G) * Vt
    coupling_flux.append(flux)
coupling_flux = np.array(coupling_flux)

fig, ax = plt.subplots()
for i in range(n):
    ax.plot(coupling_flux[:, i], label=f"district {nodes[i]}")
ax.set_xlabel("time step")
ax.set_ylabel("sum_j G_ij (V_j - V_i)")
ax.set_title("Cross-district coupling flux")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


## NetworkX conductance-weighted graph


In [ ]:
pos = nx.spring_layout(graph.graph, seed=7)
edge_widths = [3.0 * graph.get_conductance(u, v) for u, v in graph.graph.edges]

fig, ax = plt.subplots(figsize=(9, 6))
nx.draw_networkx_nodes(graph.graph, pos, node_color="#ecf0f1", edgecolors="k", node_size=600, ax=ax)
nx.draw_networkx_edges(graph.graph, pos, width=edge_widths, arrows=True, ax=ax)
nx.draw_networkx_labels(graph.graph, pos, ax=ax)
ax.set_title("District graph — edge width ~ G_ij")
ax.axis("off")
plt.tight_layout()
plt.show()


## Symmetry and zero diagonal after assembly


In [ ]:
# conductance_matrix mirrors off-diagonal; diagonal may be zero
off_diag = G[~np.eye(n, dtype=bool)]
assert off_diag.max() > 0.0
print(f"max off-diagonal G_ij = {off_diag.max():.4f}")


## Conclusions

1. `conductance_matrix` assembles a symmetric $G_{ij}$ from district portal edge attributes.
2. `district_master_step` evolves node potentials with ER=EPR coupling and branch kicks.
3. Cross-district coupling flux shows energy/information redistribution across the DAG.
4. Results validate the discrete master equation form of RBLE Eq. (7) for multiverse district networks.
